# Phase 4 — VGG-11/16 (2014)

**3×3 커널 중첩**과 **Batch Normalization**을 다룬다.

- 모델: `src/dl_history/models/vgg.py` — `make_layers(cfg)` 블록 팩토리로 VGG-11/16 구성
- 입력 적응: 28×28 → 32×32 zero padding (5번의 pooling 후 1×1이 되도록), classifier는 512로 축소
- 전체 실험 재현: `uv run python scripts/run_phase4.py`

핵심 질문 두 가지:
1. **5×5 커널 하나 대신 3×3 커널 두 개**를 쌓으면 무엇이 좋아지는가?
2. **BatchNorm**은 왜 깊은 망의 학습을 안정시키는가?


In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from dl_history.data import load_fashion_mnist_loaders
from dl_history.models.vgg import CFG_VGG11, CFG_VGG16, VGG11, VGG16, VGG11Wide, make_layers
from dl_history.train import train_model

torch.manual_seed(0)

print("VGG-11 cfg:", CFG_VGG11)
print("VGG-16 cfg:", CFG_VGG16)


## 1. `make_layers` — config 리스트로 블록을 찍어낸다

정수는 `Conv2d(3×3, padding=1)` → (BatchNorm) → ReLU, `"M"`은 2×2 MaxPool.
같은 팩토리 하나로 VGG-11과 VGG-16이 나온다 — 깊이를 **config 길이**로 표현하는 것이 VGG의 기법이다.


In [ ]:
for name, cfg in [("VGG-11", CFG_VGG11), ("VGG-16", CFG_VGG16)]:
    trunk = make_layers(cfg, batch_norm=True)
    convs = sum(1 for m in trunk if isinstance(m, nn.Conv2d))
    pools = sum(1 for m in trunk if isinstance(m, nn.MaxPool2d))
    print(f"{name}: conv {convs}개, pool {pools}개, 총 {len(trunk)}개 모듈")

padded = nn.ZeroPad2d(2)(torch.randn(2, 1, 28, 28))   # 28 + 2*2 = 32
print("trunk output:", tuple(VGG11().features(padded).shape))   # 5번 pooling → 1×1


## 2. 3×3 두 개 = 5×5 하나 (Receptive Field)

가중치를 전부 1로 둔 선형 합성곱에 **가운데 한 픽셀만 1인 입력**을 넣어보면,
3×3 두 개를 쌓은 것과 5×5 하나가 **정확히 같은 5×5 영역**에서 반응한다.


In [ ]:
stack = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Conv2d(1, 1, 3, padding=1))
wide = nn.Conv2d(1, 1, 5, padding=2)
for m in (stack[0], stack[1], wide):
    nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

impulse = torch.zeros(1, 1, 9, 9); impulse[0, 0, 4, 4] = 1.0
s, w = stack(impulse)[0, 0], wide(impulse)[0, 0]
print("3×3 두 개 반응 영역:", (s != 0).sum().item(), "픽셀")
print("5×5 하나  반응 영역:", (w != 0).sum().item(), "픽셀")
assert torch.equal(s != 0, w != 0)          # 같은 receptive field


## 3. 파라미터 수 — 18 vs 25

같은 채널 수에서 두 3×3은 `2·3·3 = 18`, 5×5 하나는 `5·5 = 25`개의 가중치를 쓴다.
즉 **같은 receptive field를 더 적은 파라미터 + 추가 비선형성으로** 얻는다.


In [ ]:
for name, model in [("VGG-11 (3×3×2)", VGG11()), ("VGG-11 wide (5×5×1)", VGG11Wide()),
                    ("VGG-16 (3×3×2, deeper)", VGG16())]:
    print(f"{name:26s} {sum(p.numel() for p in model.parameters()):>12,} params")


## 4. Exp A — 3×3 중첩 vs 5×5 하나 (학습 비교)

동일 조건(BatchNorm, 2 epoch 데모)에서 학습시킨다.
(전체 실험은 스크립트에서 8 epoch)


In [ ]:
EPOCHS = 2
loaders = load_fashion_mnist_loaders(batch_size=128)

histories_a = {}
for label, model in [("VGG-11 (3×3)", VGG11(batch_norm=True)),
                     ("VGG-11 wide (5×5)", VGG11Wide(batch_norm=True))]:
    torch.manual_seed(0)
    hist, sec = train_model(model, loaders, epochs=EPOCHS, lr=0.01, momentum=0.9)
    histories_a[label] = hist
    print(f"{label:22s} val acc {hist.val_acc[-1]:.4f}  ({sec:.1f}s/epoch)")

fig, ax = plt.subplots(figsize=(8, 5))
for label, hist in histories_a.items():
    ax.plot(hist.val_acc, marker="o", label=label)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy")
ax.set_title("Exp A — 3×3 stack vs one 5×5"); ax.grid(alpha=0.3); ax.legend()
plt.show()


## 5. Exp B — BatchNorm과 초기화

세 가지를 비교한다:

| 실행 | BatchNorm | 초기화 | 예상 |
|---|---|---|---|
| `no BN, default init` | ✗ | PyTorch 기본 | 신호 소실 → 학습 안 됨 |
| `no BN, He init` | ✗ | Kaiming | 살아나지만 BN보다 느림 |
| `+ BN` | ✓ | PyTorch 기본 | 가장 안정적 |

BatchNorm은 층마다 활성값을 다시 정규화해주므로 **초기화에 대한 의존을 없앤다**.


In [ ]:
histories_b = {}
variants = [
    ("no BN, default init", VGG11(batch_norm=False)),
    ("no BN, He init",      VGG11(batch_norm=False, init="he")),
    ("+ BN",                VGG11(batch_norm=True)),
]
for label, model in variants:
    torch.manual_seed(0)
    hist, sec = train_model(model, loaders, epochs=EPOCHS, lr=0.01, momentum=0.9)
    histories_b[label] = hist
    print(f"{label:22s} train loss {hist.train_loss[-1]:.4f}  val acc {hist.val_acc[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, hist in histories_b.items():
    axes[0].plot(hist.train_loss, marker="o", label=label)
    axes[1].plot(hist.val_acc, marker="o", label=label)
axes[0].set_title("Train loss"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("Epoch")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()
plt.show()


## 정리

- **3×3 두 개**는 5×5 하나와 같은 receptive field를 **더 적은 파라미터(18 vs 25)** 와
  추가 비선형성으로 얻는다 — VGG가 깊이를 쌓을 수 있었던 이유.
- **BatchNorm 없는 깊은 VGG**는 기본 초기화에서 신호가 죽어 학습이 시작되지 않는다.
  He 초기화로 살릴 수 있지만, BatchNorm을 쓰면 초기화를 신경 쓸 필요가 없다.
- 다음 Phase 5(ResNet)에서는 층을 더 깊게 쌓았을 때의 **Degradation** 문제와
  skip connection을 다룬다.
